# AI-aided SPI driver and recovered-core testbench generation


## 1. Setup


### 1.1. Repository and test target


In [1]:
# locate the checkout from its root or tb directory.
import json
import os
from pathlib import Path
import subprocess
import sys

# allow an explicit checkout path when Jupyter starts elsewhere.
start = Path(os.environ.get("AHA_REPO_ROOT", Path.cwd())).expanduser().resolve()
ROOT = next((p for p in (start, *start.parents)
             if (p / "docs/transaction_specification.md").is_file()), None)
# fail early if the notebook cannot find its source inputs.
if ROOT is None:
    raise FileNotFoundError("Set AHA_REPO_ROOT to the aha2026-dp1 checkout")
sys.path.insert(0, str(ROOT))
from scripts.testbench_workflow import (
    AILogger, DEFAULT_BASE_URL, DEFAULT_MODEL, LLMClient, build_prompt, discover_tools,
    file_hash, repair_prompt, request_generation, run_simulation,
    validate_artifacts, write_artifacts,
)
from scripts.driver_workflow import (
    build_driver_prompt, discover_driver_tools, request_mock, run_driver, write_mock,
)

# select the independent driver test unless recovered-core validation is requested.
TARGET = os.environ.get("AHA_TEST_TARGET", "tb_driver")
if TARGET not in {"tb_driver", "crypto_tb"}:
    raise ValueError("AHA_TEST_TARGET must be tb_driver or crypto_tb")
tools = discover_driver_tools() if TARGET == "tb_driver" else discover_tools()
print("Repository:", ROOT)
print("Target:", TARGET)
print("Toolchain:", json.dumps(tools, indent=2))
print("DRIVER TEST: mock success validates the harness, not the recovered cryptographic circuit."
      if TARGET == "tb_driver" else "RECOVERED DUT TEST: supplied application comparisons.")


Repository: /home/bluto/Desktop/_ece4313/hardwareChallenge/aha2026-dp1
Target: tb_driver
Toolchain: {
  "iverilog": "/home/bluto/.local/opt/oss-cad-suite-2026-11-05/bin/iverilog",
  "vvp": "/home/bluto/.local/opt/oss-cad-suite-2026-11-05/bin/vvp"
}
DRIVER TEST: mock success validates the harness, not the recovered cryptographic circuit.


### 1.2. Reproducible run settings


In [2]:
# preserve the lab's Portkey settings without storing a credential in the notebook.
os.environ.setdefault("PORTKEY_BASE_URL", DEFAULT_BASE_URL)
PORTKEY_API_KEY = os.environ.get("PORTKEY_API_KEY", "")

# choose replay explicitly until live generation is requested.
MODE = os.environ.get("AHA_GENERATION_MODE", "replay")
MODEL = os.environ.get("PORTKEY_MODEL", DEFAULT_MODEL)
SEED = 4313
RANDOM_TRIALS = 8
API_TIMEOUT_S = 120
SIMULATION_TIMEOUT_S = 30
MAX_ATTEMPTS = 3 if TARGET == "crypto_tb" and MODE == "live" else 1
TEMPERATURE = 0.0

# route transcripts by the notebook folder, independent of the kernel directory.
logger = AILogger(ROOT, source_dir=ROOT / "tb")

# allocate fresh generated artifacts for every run.
RUN_DIR = ROOT / "results/simulation" / logger.run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
print(json.dumps({"mode": MODE, "model": MODEL if MODE == "live" else "recorded Codex generation",
                  "target": TARGET, "simulation_seed": SEED, "random_trials": RANDOM_TRIALS,
                  "max_attempts": MAX_ATTEMPTS, "ai_logs": str(logger.run_dir),
                  "artifacts": str(RUN_DIR)}, indent=2))
# the seed fixes stimuli; a live model response is not guaranteed to repeat.
print("Live mode requires PORTKEY_API_KEY; credentials are never printed or logged.")


{
  "mode": "replay",
  "model": "recorded Codex generation",
  "target": "tb_driver",
  "simulation_seed": 4313,
  "random_trials": 8,
  "max_attempts": 1,
  "ai_logs": "/home/bluto/Desktop/_ece4313/hardwareChallenge/aha2026-dp1/ai_logs/tb/runs/20261106T023151.830169Z-ee5707ea",
  "artifacts": "/home/bluto/Desktop/_ece4313/hardwareChallenge/aha2026-dp1/results/simulation/20261106T023151.830169Z-ee5707ea"
}
Live mode requires PORTKEY_API_KEY; credentials are never printed or logged.


## 2. Sources and AI transcripts


### 2.1. Source fingerprints


In [3]:
# fingerprint only the selected target's inputs; driver tests need no recovered files.
SOURCE_PATHS = ["tb/spi_operations.svh", "scripts/testbench_workflow.py"]
if TARGET == "tb_driver":
    SOURCE_PATHS += ["tb/tb_driver.sv", "scripts/driver_workflow.py",
                     "prompts/generate_spi_mock.txt"]
else:
    SOURCE_PATHS += ["docs/transaction_specification.md", "inputs/spi_ice40_crypto_ip_test.py",
                     "recovery/recovered_sim.v", "prompts/generate_spi_testbench.txt"]
source_hashes = {path: file_hash(ROOT / path) for path in SOURCE_PATHS}
revision = subprocess.run(["git", "rev-parse", "HEAD"], cwd=ROOT, capture_output=True,
                          text=True, check=True, timeout=10).stdout.strip()
logger.log_event("context", source_sha256=source_hashes, revision=revision,
                 seed=SEED, mode=MODE, target=TARGET)
print(json.dumps(source_hashes, indent=2))


{
  "tb/spi_operations.svh": "69c135eb26bf4696ce80be6e0d4af1e3d5d2059eee8fa46d8f78383865801fe1",
  "scripts/testbench_workflow.py": "0f832e2d55a576819ac9446c32c7c232cf6d32c3b4e70a37b869720ef2f17e1a",
  "tb/tb_driver.sv": "bc815f53d43b95a868319582bf252c98a0471e1a928821996eceb9fc9855a66f",
  "scripts/driver_workflow.py": "5825499ff8cbd46436bd8489bb1ea16959226012ff604880f17911dade74e73a",
  "prompts/generate_spi_mock.txt": "0eeb70a7db61d40a1eabf15f9e124ce5bc4c96f20275c40f133b620946c1ffa4"
}


### 2.2. Portkey client and automatic logging


In [4]:
# reuse the lab-style SDK client across generation and repair attempts.
llm_client = LLMClient(api_key=PORTKEY_API_KEY, model=MODEL, provider="portkey",
                       logger=logger, timeout_s=API_TIMEOUT_S)
print("Portkey API key configured:", llm_client.is_available())

# persist each prompt and response, including the temperature-specific retry.
def ask_ai(prompt):
    return request_generation(prompt, logger, mode=MODE, model=MODEL,
                              temperature=TEMPERATURE, seed=SEED,
                              timeout_s=API_TIMEOUT_S, client=llm_client)

# replay remains explicit and does not contact Portkey.
print("Transcript:", logger.events_path)
print("Replay reads a recorded AI response; live mode uses LLMClient.generate().")


Portkey API key configured: False
Transcript: /home/bluto/Desktop/_ece4313/hardwareChallenge/aha2026-dp1/ai_logs/tb/runs/20261106T023151.830169Z-ee5707ea/events.jsonl
Replay reads a recorded AI response; live mode uses LLMClient.generate().


## 3. Independent driver test


### 3.1. Mock generation prompt


In [5]:
# generate a protocol fixture without a cryptographic oracle or recovered source.
if TARGET == "tb_driver":
    driver_prompt = build_driver_prompt(ROOT)
    print(driver_prompt)
else:
    print("DRIVER TEST skipped: select AHA_TEST_TARGET=tb_driver to test the harness independently.")


Generate an AI-authored simulation-only serial mock for the attached trusted
SPI driver harness. This is the tb_driver target: it validates the harness's bit
ordering, CS framing, and scoreboard. It cannot establish that a recovered
cryptographic circuit works. Do not implement AES or any crypto transformation.
Do not modify the harness, shared operations, or a recovered DUT.

Return exactly one JSON object with string fields mock_sv and
scheduling_explanation. The module must be named spi_mock_peripheral with inputs
SCK, RST_N, MOSI, NORM_CS_N, START, ENC_DEC, and [31:0] RESPONSE_WORD; outputs
MISO, BUSY, ICE_LED, [31:0] RECEIVED_WORD, [31:0] FRAME_COUNT, [31:0] FRAME_BITS.

RESPONSE_WORD is test-only configuration latched at CS assertion, independent of
MOSI. Shift its MSB onto registered MISO at each selected rising SCK. Shift MOSI
MSB first into RECEIVED_WORD. Require exactly 32 rising edges per uninterrupted
CS-low frame; reject short, split, and overlong selections. CS changes mu

### 3.2. Generate the mock and run tb_driver


In [6]:
# log the mock request and response before compiling with the trusted driver operations.
if TARGET == "tb_driver":
    driver_generated_dir = RUN_DIR / "tb_driver/generated"
    driver_output_dir = RUN_DIR / "tb_driver/simulation"
    # retain errors and stop immediately if generation or driver validation fails.
    try:
        mock_artifacts = request_mock(driver_prompt, logger, mode=MODE, client=llm_client,
                                      seed=SEED, timeout_s=API_TIMEOUT_S)
        mock_paths = write_mock(mock_artifacts, driver_generated_dir)
        logger.log_event("validation", target=TARGET, status="mock source accepted",
                         artifact_sha256={key: file_hash(path) for key, path in mock_paths.items()})
        driver_result = run_driver(ROOT, driver_output_dir, seed=SEED,
                                   timeout_s=SIMULATION_TIMEOUT_S, source_dir=driver_generated_dir)
        logger.log_event("simulation", target=TARGET, result=driver_result)
        print((driver_output_dir / "simulation.log").read_text())
    except Exception as error:
        logger.log_event("validation", target=TARGET, status="failed", message=str(error))
        raise


DRIVER TEST tb_driver: mock validates harness only, not recovered cryptography
VCD info: dumpfile waveforms.vcd opened for output.
DRIVER TEST configuration seed=4313 scenario=normal
PASS driver.reset busy=0
DRIVER TEST driver.frame.0 tx=01234567 response=d2963a5c
TRANSFER driver.frame.0 mosi=01234567 miso=d2963a5c
DRIVER FRAME number=1 bits=32 mosi=01234567
PASS driver.frame.0.rx value=d2963a5c
PASS driver.frame.0.tx value=01234567
PASS driver.frame.0.wire value=01234567
PASS driver.frame.0.frames value=00000001
PASS driver.frame.0.bits value=00000020
DRIVER TEST driver.frame.1 tx=00000000 response=d3943e54
TRANSFER driver.frame.1 mosi=00000000 miso=d3943e54
DRIVER FRAME number=2 bits=32 mosi=00000000
PASS driver.frame.1.rx value=d3943e54
PASS driver.frame.1.tx value=00000000
PASS driver.frame.1.wire value=00000000
PASS driver.frame.1.frames value=00000002
PASS driver.frame.1.bits value=00000020
DRIVER TEST driver.frame.2 tx=ffffffff response=d092324c
TRANSFER driver.frame.2 mosi=ffff

### 3.3. Driver scheduling and results


In [7]:
# report mock evidence separately from recovered-circuit comparisons.
if TARGET == "tb_driver":
    assert driver_result["passed"] and driver_result["test_kind"] == "driver"
    print(mock_artifacts["scheduling_explanation"])
    print("DRIVER TEST result:")
    print(json.dumps(driver_result, indent=2))
    print("DRIVER TEST waveforms:", driver_result["waveform"])
    print("The mock checks bit ordering, chip-select framing, and the scoreboard; "
          "it does not establish that the recovered cryptographic circuit works.")


This is a simulation-only serial peripheral, not a cryptographic reference. Its RESPONSE_WORD is configured independently of MOSI. The trusted harness compares its receive result against that asymmetric word, checks mock MOSI telemetry against an independent expected word, and reconstructs MOSI with a separate passive wire monitor. Both monitors require exactly 32 selected SCK rising edges per frame; clocks with CS high cannot shift serial state. Readout deliberately transmits 0x96a50fc3 and verifies that input capture too.

Only initialize_interface sets SCK low at time zero; sck_pulse owns every later transition. Controls and MOSI are driven while SCK is low, a 10000 ns half-period provides setup, and the mock samples at the next rising edge. Its MISO assignment uses nonblocking assignment, so the shared driver samples 1 ns after the rising edge, following NBA updates. No same-edge testbench input changes race the peripheral. CS deasserts after SCK falls; the harness waits 1 ns befor

## 4. Optional recovered-core test


### 4.1. Recovered-core generation prompt


In [8]:
# use the application contract only when the recovered target is selected.
if TARGET == "crypto_tb":
    generation_prompt = build_prompt(ROOT)
    print(generation_prompt)
else:
    print("RECOVERED DUT TEST skipped: select AHA_TEST_TARGET=crypto_tb when its source is available.")


RECOVERED DUT TEST skipped: select AHA_TEST_TARGET=crypto_tb when its source is available.


### 4.2. Bounded generation and validation attempts


In [9]:
# keep recovered-core compilation separate from the driver-only target.
if TARGET == "crypto_tb":
    # validate every candidate against the recovered DUT before accepting it.
    prompt = generation_prompt
    artifacts = None
    result = None
    attempt_history = []
    # bound generation attempts and retain each failure for repair.
    for attempt in range(1, MAX_ATTEMPTS + 1):
        artifacts = None
        attempt_dir = RUN_DIR / f"attempt-{attempt:02d}"
        generated_dir = attempt_dir / "generated"
        simulation_dir = attempt_dir / "simulation"
        # preserve generation, compilation, and simulation failures as repair evidence.
        try:
            artifacts = ask_ai(prompt)
            validate_artifacts(artifacts)
            paths = write_artifacts(artifacts, generated_dir)
            logger.log_event("validation", attempt=attempt, status="source accepted",
                             artifact_sha256={key: file_hash(path) for key, path in paths.items()})
            result = run_simulation(ROOT, generated_dir, simulation_dir, seed=SEED,
                                    random_trials=RANDOM_TRIALS, timeout_s=SIMULATION_TIMEOUT_S)
            logger.log_event("simulation", attempt=attempt, result=result)
            attempt_history.append({"attempt": attempt, "passed": True})
            print((simulation_dir / "simulation.log").read_text())
            break
        except Exception as error:
            event = logger.log_event("validation", attempt=attempt, status="failed", message=str(error))
            attempt_history.append({"attempt": attempt, "passed": False, "error": event["message"]})
            # stop at the attempt limit; replay never falls back to an API request.
            if attempt == MAX_ATTEMPTS:
                raise RuntimeError(f"Validation failed; inspect {logger.run_dir} and {attempt_dir}") from None
            diagnostics = event["message"]
            # send bounded compiler/simulator excerpts rather than an entire waveform.
            for filename in ("compile.log", "simulation.log"):
                path = simulation_dir / filename
                # include only diagnostics that actually exist for this attempt.
                if path.is_file():
                    diagnostics += "\n" + filename + "\n" + path.read_text()[-8000:]
            # retain malformed responses as feedback when no valid artifact exists.
            events = [json.loads(line) for line in logger.events_path.read_text().splitlines()]
            request_id = next(event["request_id"] for event in reversed(events) if event["kind"] == "prompt")
            outputs = [event for event in events if event["kind"] == "output"
                       and event["request_id"] == request_id]
            previous = artifacts or {"raw_response": outputs[-1]["raw"] if outputs else "no response"}
            prompt = repair_prompt(generation_prompt, previous, diagnostics)


### 4.3. Recovered-core scheduling explanation


In [10]:
# show recovered-core scheduling only after its actual simulation.
if TARGET == "crypto_tb":
    print(artifacts["scheduling_explanation"])
    print("Saved explanation:", paths["scheduling_explanation"])


### 4.4. Recovered-core completion observations


In [11]:
# preserve measured recovered-core timing independently of the mock.
if TARGET == "crypto_tb":
    observations = []
    # compare completion traces without shortening the supplied clock sequence.
    for mode in ("encrypt", "decrypt"):
        samples = [(int(edge), int(busy)) for name, edge, busy in result["processing_trace"] if name == mode]
        idle_edges = [edge for edge, busy in samples if busy == 0]
        observations.append({"operation": mode, "busy_by_edge": samples,
                             "first_observed_idle_edge": min(idle_edges) if idle_edges else None})
    logger.log_event("measurements", observations=observations)
    print(json.dumps(observations, indent=2))
    # BUSY timing does not by itself prove the internal parallel-load edge.
    print("The first readout occurs after e7; earlier result availability needs separate trials.")


## 5. Source integrity and run summary


In [12]:
# require unchanged selected inputs before accepting the recorded result.
assert all(file_hash(ROOT / path) == digest for path, digest in source_hashes.items())
if TARGET == "tb_driver":
    final_result = driver_result
    assert final_result["passed"] and final_result["test_kind"] == "driver"
    logger.log_event("summary", target=TARGET, result=final_result)
else:
    final_result = result
    assert final_result["passed"] and final_result["dut_unchanged"]
    logger.log_event("summary", target=TARGET, attempts=attempt_history, result=final_result)
print("DRIVER TEST PASS" if TARGET == "tb_driver" else "RECOVERED DUT TEST PASS")
print("Open this VCD in GTKWave:", final_result["waveform"])
print("AI prompts and responses:", logger.events_path)


DRIVER TEST PASS
Open this VCD in GTKWave: /home/bluto/Desktop/_ece4313/hardwareChallenge/aha2026-dp1/results/simulation/20261106T023151.830169Z-ee5707ea/tb_driver/simulation/waveforms.vcd
AI prompts and responses: /home/bluto/Desktop/_ece4313/hardwareChallenge/aha2026-dp1/ai_logs/tb/runs/20261106T023151.830169Z-ee5707ea/events.jsonl
